
# AI Agent for Geospatial Outlet Clustering
### Haversine Distance · DBSCAN · QGIS Export · smolagents · Gemini 2.5 Flash

**Author:** Kate Lawal
**Origin:** Updated from `haversine_vector_distance.py` (2020)
**Purpose:** Cluster retail outlets by proximity using Haversine distance,
             with QGIS-ready exports for visual QA and spatial audit.

---

## Overview

This notebook converts a 2020 Python script for haversine distance-based outlet
clustering into a **multi-tool AI agent**. The original script had several bugs
(wrong loop signature, undefined variables, misaligned indentation, leftover test code)
and no geographic visualisation or QGIS integration.

The agent below fixes all of that and adds:

```
┌─────────────────────────────────────────────────────────────────────┐
│              OUTLET CLUSTERING AGENT PIPELINE                        │
│                                                                      │
│  USER: "Cluster these outlets within 1500ft of each other"          │
│                                                                      │
│  TOOL 1: load_and_validate_outlets()                                │
│          → Loads CSV, validates lat/lon, flags missing coords        │
│                                                                      │
│  TOOL 2: compute_haversine_distance_matrix()                        │
│          → Pairwise Haversine distances (miles & feet)              │
│          → Replaces the broken original loop                        │
│                                                                      │
│  TOOL 3: cluster_outlets_dbscan()                                   │
│          → DBSCAN clustering on Haversine distances                 │
│          → Labels every outlet: cluster ID or noise                 │
│                                                                      │
│  TOOL 4: summarise_clusters()                                       │
│          → Cluster statistics: size, centroid, spread               │
│                                                                      │
│  TOOL 5: plot_clusters_map()                                        │
│          → Inline scatter map, colour-coded by cluster              │
│                                                                      │
│  TOOL 6: export_for_qgis()                                         │
│          → GeoPackage (.gpkg) + CSV + QGIS layer style (.qml)      │
│          → Ready to drag-and-drop into QGIS                        │
│                                                                      │
│  TOOL 7: export_distance_pairs()                                    │
│          → Replaces clustered_outlier.csv from original script      │
│          → All pairs within threshold, with unique cluster IDs      │
│                                                                      │
│  OUTPUT: Clustered outlets + QGIS project files                     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## What Was Wrong in the Original Script

| Bug | Location | Fix Applied |
|-----|----------|-------------|
| `range(len(colors, 0))` — `len()` takes 1 arg | Line 50 | `range(len(colors))` |
| `colours[i]` — typo (`colours` vs `colors`) | Line 52 | `colors.iloc[i]` |
| `unique=[]` inside loop resets on every iteration | Line 51 | Moved outside loop |
| `nc['unique_id'] = nc.id2.map(unique_dict)` — wrong indent (inside `else`) | Line 57 | Restructured entirely |
| `haversine_vector(source, destination, Unit.FEET)` — undefined variables | Line 40 | Replaced with `scikit-learn` Haversine |
| `np.append(unique, ratios[0])` before loop defines `unique` | Line 48 | Removed; initialised correctly |
| Presidents loop — unrelated test code | Lines 66-68 | Removed |
| Hard-coded Windows file paths | Lines 14-15, 43 | Replaced with parameters |
| No QGIS integration | Entire script | Added via geopandas + .qml export |
| No visualisation | Entire script | Added matplotlib + folium map |


---

## Setup

Run once. Restart runtime if prompted.

In [ ]:

!pip install smolagents scikit-learn geopandas pandas numpy matplotlib              folium pillow scipy -q


In [ ]:

from smolagents import CodeAgent, tool
from smolagents import OpenAIServerModel
from smolagents.monitoring import LogLevel

import pandas as pd
import numpy as np
from sklearn.neighbors import DistanceMetric
from sklearn.cluster import DBSCAN
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import geopandas as gpd
from shapely.geometry import Point
import io, json, os, warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported.")


### API Key

In [ ]:

import getpass
API_KEY = getpass.getpass("Enter your Gemini API key: ")


In [ ]:

# from google.colab import userdata
# API_KEY = userdata.get("GEMINI_API_KEY")


In [ ]:

model = OpenAIServerModel(
    model_id="gemini-2.5-flash",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=API_KEY,
)
print("✅ Model: gemini-2.5-flash")



---

## Generate Sample Data

If you don't have your own CSV yet, run this cell to create realistic
sample outlet data in the format the tools expect.
Replace with your own file at any time — just match the column names.

**Required columns:** `RESPONSEID`, `GPS_LAT_DD`, `GPS_LOG_DD`


In [ ]:

def generate_sample_outlets(n_outlets=120, n_true_clusters=8, seed=42):
    """
    Generate synthetic outlet data around Lagos, Nigeria
    with known ground-truth clusters for testing.
    """
    rng = np.random.default_rng(seed)

    # Cluster centres around Lagos (lat, lon)
    centres = [
        (6.4530,  3.3958),   # Victoria Island
        (6.5244,  3.3792),   # Ikeja
        (6.4698,  3.5852),   # Lekki Phase 1
        (6.4355,  3.4197),   # Surulere
        (6.6018,  3.3515),   # Agege
        (6.3750,  3.5667),   # Ajah
        (6.5958,  3.4536),   # Ketu
        (6.4480,  3.4137),   # Yaba
    ]

    rows = []
    outlet_id = 1

    for cluster_idx, (clat, clon) in enumerate(centres):
        n = rng.integers(8, 22)                          # outlets per cluster
        for _ in range(n):
            # ~50–800m spread around cluster centre (degrees)
            lat = clat + rng.normal(0, 0.004)
            lon = clon + rng.normal(0, 0.004)
            rows.append({
                "RESPONSEID":  f"OUT{outlet_id:04d}",
                "GPS_LAT_DD":  round(lat, 6),
                "GPS_LOG_DD":  round(lon, 6),
                "OUTLET_NAME": f"Outlet_{outlet_id:04d}",
                "CHANNEL":     rng.choice(["Supermarket","Pharmacy","HORECA","Convenience"]),
                "TRUE_CLUSTER": cluster_idx,
            })
            outlet_id += 1

    # Add some noise outlets far from clusters
    for _ in range(10):
        rows.append({
            "RESPONSEID":   f"OUT{outlet_id:04d}",
            "GPS_LAT_DD":   round(rng.uniform(6.35, 6.65), 6),
            "GPS_LOG_DD":   round(rng.uniform(3.30, 3.70), 6),
            "OUTLET_NAME":  f"Outlet_{outlet_id:04d}",
            "CHANNEL":      rng.choice(["Supermarket","Pharmacy","HORECA","Convenience"]),
            "TRUE_CLUSTER": -1,
        })
        outlet_id += 1

    df = pd.DataFrame(rows)
    df.to_csv("sample_outlets.csv", index=False)
    print(f"✅ sample_outlets.csv created — {len(df)} outlets, "
          f"{n_true_clusters} true clusters + noise")
    return df

sample_df = generate_sample_outlets()
sample_df.head(10)



---

# Part 1: The Clustering Tools

Seven tools, each doing one job precisely.
The agent combines them dynamically based on the question asked.



---

## Tool 1: Load & Validate Outlet Data

**What this fixes:** The original script used hard-coded Windows paths and had no
validation — any bad row would silently corrupt the distance matrix.

This tool validates coordinates, flags problematic rows, and returns a clean summary.


In [ ]:

@tool
def load_and_validate_outlets(filepath: str) -> str:
    """
    Loads an outlet CSV file, validates GPS coordinates, and returns a
    data quality summary. Use this as the FIRST step before any distance
    or clustering calculation. The file must contain columns:
    RESPONSEID, GPS_LAT_DD, GPS_LOG_DD.

    Args:
        filepath: Path to the CSV file containing outlet GPS coordinates

    Returns:
        JSON string with record counts, validation flags, coordinate bounds,
        and a list of any rows dropped due to missing or invalid coordinates.
    """
    if not os.path.exists(filepath):
        return json.dumps({"error": f"File not found: {filepath}"})

    df = pd.read_csv(filepath)

    required = ['RESPONSEID', 'GPS_LAT_DD', 'GPS_LOG_DD']
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        return json.dumps({"error": f"Missing required columns: {missing_cols}"})

    total_loaded = len(df)

    # Drop rows with missing coordinates
    df_clean = df.dropna(subset=['GPS_LAT_DD', 'GPS_LOG_DD']).copy()
    dropped_missing = total_loaded - len(df_clean)

    # Flag out-of-range coordinates
    invalid_lat = df_clean[(df_clean['GPS_LAT_DD'] < -90) | (df_clean['GPS_LAT_DD'] > 90)]
    invalid_lon = df_clean[(df_clean['GPS_LOG_DD'] < -180) | (df_clean['GPS_LOG_DD'] > 180)]
    df_clean = df_clean[
        (df_clean['GPS_LAT_DD'].between(-90, 90)) &
        (df_clean['GPS_LOG_DD'].between(-180, 180))
    ]
    dropped_invalid = len(invalid_lat) + len(invalid_lon)

    # Save cleaned file for downstream tools
    clean_path = filepath.replace('.csv', '_clean.csv')
    df_clean.to_csv(clean_path, index=False)

    return json.dumps({
        "filepath":           filepath,
        "clean_filepath":     clean_path,
        "total_loaded":       total_loaded,
        "valid_outlets":      len(df_clean),
        "dropped_missing":    dropped_missing,
        "dropped_invalid":    dropped_invalid,
        "coordinate_bounds":  {
            "lat_min":  round(df_clean['GPS_LAT_DD'].min(), 6),
            "lat_max":  round(df_clean['GPS_LAT_DD'].max(), 6),
            "lon_min":  round(df_clean['GPS_LOG_DD'].min(), 6),
            "lon_max":  round(df_clean['GPS_LOG_DD'].max(), 6),
        },
        "columns_available":  list(df_clean.columns),
        "sample_ids":         df_clean['RESPONSEID'].head(5).tolist(),
    }, indent=2)

print("Tool 1: load_and_validate_outlets")
r = json.loads(load_and_validate_outlets("sample_outlets.csv"))
print(f"  Loaded: {r['total_loaded']} | Valid: {r['valid_outlets']} | Dropped: {r['dropped_missing']}")
print(f"  Bounds: lat [{r['coordinate_bounds']['lat_min']}, {r['coordinate_bounds']['lat_max']}]")



---

## Tool 2: Haversine Distance Matrix

**What this fixes:** The original script computed the distance matrix correctly
but then broke trying to unpack it with a buggy loop. This tool computes the full
pairwise matrix cleanly and saves it — no loop needed downstream.

**Key fix:** `sklearn.neighbors.DistanceMetric` with `'haversine'` requires radians.
We convert degrees → radians here, multiply by 3959 (Earth radius in miles) or
20902640 (Earth radius in feet) depending on the unit requested.


In [ ]:

@tool
def compute_haversine_distance_matrix(
    filepath: str,
    second_filepath: str = None,
    unit: str = "miles",
) -> str:
    """
    Computes the pairwise Haversine distance matrix between outlets.
    If only one file is provided, computes all-vs-all distances within that file.
    If two files are provided, computes cross-distances (file1 outlets vs file2 outlets),
    exactly replicating the original haversine_vector_distance.py intent.
    Saves the distance matrix as a long-format CSV (pairs + distance).

    Args:
        filepath:        Path to the primary outlet CSV (must have GPS_LAT_DD, GPS_LOG_DD, RESPONSEID)
        second_filepath: Optional path to a second outlet CSV for cross-distance calculation
        unit:            Distance unit - 'miles', 'feet', or 'km' (default 'miles')

    Returns:
        JSON string with matrix shape, summary statistics, threshold counts, and output file path.
    """
    EARTH_RADIUS = {"miles": 3959.0, "feet": 20902640.0, "km": 6371.0}
    if unit not in EARTH_RADIUS:
        return json.dumps({"error": f"unit must be one of {list(EARTH_RADIUS.keys())}"})

    df1 = pd.read_csv(filepath).dropna(subset=['GPS_LAT_DD','GPS_LOG_DD'])

    # Convert degrees to radians — required by sklearn Haversine
    df1['lat_rad'] = np.radians(df1['GPS_LAT_DD'])
    df1['lon_rad'] = np.radians(df1['GPS_LOG_DD'])

    if second_filepath and os.path.exists(second_filepath):
        df2 = pd.read_csv(second_filepath).dropna(subset=['GPS_LAT_DD','GPS_LOG_DD'])
        df2['lat_rad'] = np.radians(df2['GPS_LAT_DD'])
        df2['lon_rad'] = np.radians(df2['GPS_LOG_DD'])
        id_col2 = 'RESPONSEID_2' if 'RESPONSEID_2' in df2.columns else 'RESPONSEID'
    else:
        df2     = df1.copy()
        id_col2 = 'RESPONSEID'

    dist_metric = DistanceMetric.get_metric('haversine')
    matrix = dist_metric.pairwise(
        df1[['lat_rad','lon_rad']].values,
        df2[['lat_rad','lon_rad']].values,
    ) * EARTH_RADIUS[unit]

    # Convert to long format (pairs) — fixes the buggy melt/unpivot from original
    df_matrix = pd.DataFrame(
        matrix,
        index=df1['RESPONSEID'].values,
        columns=df2[id_col2].values,
    )
    df_long = df_matrix.reset_index().melt(id_vars='index')
    df_long.columns = ['RESPONSEID', 'RESPONSEID_2', f'distance_{unit}']

    # Remove self-pairs (distance == 0)
    df_long = df_long[df_long[f'distance_{unit}'] > 0].copy()

    # Threshold counts (1500 ft = 0.284 miles = 0.457 km)
    thresh = {"miles": 0.284, "feet": 1500.0, "km": 0.457}[unit]
    within_thresh = df_long[df_long[f'distance_{unit}'] <= thresh]

    out_path = filepath.replace('.csv','_distance_pairs.csv')
    df_long.to_csv(out_path, index=False)

    return json.dumps({
        "matrix_shape":         [len(df1), len(df2)],
        "total_pairs":          len(df_long),
        "unit":                 unit,
        "earth_radius_used":    EARTH_RADIUS[unit],
        "distance_stats": {
            "mean":   round(df_long[f'distance_{unit}'].mean(), 4),
            "median": round(df_long[f'distance_{unit}'].median(), 4),
            "min":    round(df_long[f'distance_{unit}'].min(), 4),
            "max":    round(df_long[f'distance_{unit}'].max(), 4),
        },
        "pairs_within_1500ft_threshold": len(within_thresh),
        "output_csv":           out_path,
        "note": (
            "Distance matrix saved as long-format CSV. "
            "Use cluster_outlets_dbscan() for cluster assignments."
        ),
    }, indent=2)

print("Tool 2: compute_haversine_distance_matrix")
r = json.loads(compute_haversine_distance_matrix("sample_outlets.csv", unit="miles"))
print(f"  Matrix: {r['matrix_shape']} | Total pairs: {r['total_pairs']}")
print(f"  Within 1500ft threshold: {r['pairs_within_1500ft_threshold']}")
print(f"  Distance stats (miles): {r['distance_stats']}")



---

## Tool 3: DBSCAN Clustering on Haversine Distances

**What this adds:** The original script only filtered pairs within a threshold —
it didn't assign proper cluster IDs. This tool runs **DBSCAN** (Density-Based
Spatial Clustering of Applications with Noise) directly on Haversine distances,
which is the correct algorithm for geographic outlet clustering because:

- It doesn't require specifying the number of clusters upfront
- It identifies noise outlets that don't belong to any cluster
- The `eps` parameter maps directly to your 1500ft (0.284 mile) threshold
- `min_samples` controls minimum outlets per cluster

**This completely replaces the broken loop logic in the original script.**


In [ ]:

@tool
def cluster_outlets_dbscan(
    filepath: str,
    eps_miles: float = 0.284,
    min_samples: int = 2,
) -> str:
    """
    Applies DBSCAN clustering to outlet GPS coordinates using Haversine distance.
    Assigns every outlet a cluster_id (-1 means noise/unclustered).
    This replaces the broken for-loop cluster assignment in the original script.
    eps_miles corresponds to the minimum inter-outlet distance threshold
    (default 0.284 miles = 1500 feet, matching the original script).

    Args:
        filepath:    Path to outlet CSV with GPS_LAT_DD and GPS_LOG_DD columns
        eps_miles:   Maximum distance between two outlets to be in the same cluster
                     in miles (default 0.284 = 1500 feet)
        min_samples: Minimum number of outlets to form a cluster (default 2)

    Returns:
        JSON string with cluster assignments, cluster count, noise count,
        and path to the output CSV with cluster_id column added.
    """
    df = pd.read_csv(filepath).dropna(subset=['GPS_LAT_DD','GPS_LOG_DD']).copy()

    # Convert to radians for sklearn Haversine metric
    coords_rad = np.radians(df[['GPS_LAT_DD','GPS_LOG_DD']].values)

    # eps must be in radians for haversine metric in sklearn
    # miles / earth_radius_miles = radians
    eps_rad = eps_miles / 3959.0

    db = DBSCAN(
        eps=eps_rad,
        min_samples=min_samples,
        algorithm='ball_tree',
        metric='haversine',
    ).fit(coords_rad)

    df['cluster_id'] = db.labels_

    # Build cluster summary using a loop
    cluster_summary = {}
    for cid in sorted(df['cluster_id'].unique()):
        if cid == -1:
            continue
        subset = df[df['cluster_id'] == cid]
        cluster_summary[int(cid)] = {
            "outlet_count":  len(subset),
            "centroid_lat":  round(subset['GPS_LAT_DD'].mean(), 6),
            "centroid_lon":  round(subset['GPS_LOG_DD'].mean(), 6),
            "outlet_ids":    subset['RESPONSEID'].tolist(),
        }

    out_path = filepath.replace('.csv', '_clustered.csv')
    df.to_csv(out_path, index=False)

    n_clusters = len([c for c in df['cluster_id'].unique() if c != -1])
    n_noise    = (df['cluster_id'] == -1).sum()

    return json.dumps({
        "filepath":         filepath,
        "output_clustered": out_path,
        "eps_miles":        eps_miles,
        "eps_feet":         round(eps_miles * 5280, 0),
        "min_samples":      min_samples,
        "total_outlets":    len(df),
        "n_clusters":       n_clusters,
        "n_clustered_outlets": int((df['cluster_id'] != -1).sum()),
        "n_noise_outlets":  int(n_noise),
        "cluster_summary":  cluster_summary,
    }, indent=2)

print("Tool 3: cluster_outlets_dbscan")
r = json.loads(cluster_outlets_dbscan("sample_outlets.csv"))
print(f"  Clusters found: {r['n_clusters']} | Clustered outlets: {r['n_clustered_outlets']} | Noise: {r['n_noise_outlets']}")



---

## Tool 4: Cluster Statistics Summary

**Concept:** After clustering, an analyst needs to understand the size distribution,
geographic spread, and average internal distances of each cluster before acting on them.
This tool produces the summary table you'd normally build manually in Excel.


In [ ]:

@tool
def summarise_clusters(clustered_filepath: str) -> str:
    """
    Produces a statistical summary of each cluster from a clustered outlet CSV.
    Calculates cluster size, centroid, geographic spread (std dev of lat/lon),
    and flags clusters that are unusually large or small.
    Use this after cluster_outlets_dbscan() to understand cluster quality.

    Args:
        clustered_filepath: Path to CSV output from cluster_outlets_dbscan()
                            (must contain cluster_id, GPS_LAT_DD, GPS_LOG_DD columns)

    Returns:
        JSON string with per-cluster statistics and overall summary.
    """
    df = pd.read_csv(clustered_filepath)
    if 'cluster_id' not in df.columns:
        return json.dumps({"error": "cluster_id column not found. Run cluster_outlets_dbscan first."})

    df_clusters = df[df['cluster_id'] != -1].copy()
    summary_rows = []

    for cid in sorted(df_clusters['cluster_id'].unique()):
        sub = df_clusters[df_clusters['cluster_id'] == cid]

        # Compute intra-cluster spread using Haversine on centroid pairs
        coords_rad = np.radians(sub[['GPS_LAT_DD','GPS_LOG_DD']].values)
        if len(sub) > 1:
            dm = DistanceMetric.get_metric('haversine')
            pairs = dm.pairwise(coords_rad) * 5280  # convert to feet
            # Upper triangle only (avoid double counting)
            upper = pairs[np.triu_indices(len(sub), k=1)]
            max_internal_ft = round(upper.max(), 1)
            avg_internal_ft = round(upper.mean(), 1)
        else:
            max_internal_ft = 0.0
            avg_internal_ft = 0.0

        summary_rows.append({
            "cluster_id":       int(cid),
            "outlet_count":     len(sub),
            "centroid_lat":     round(sub['GPS_LAT_DD'].mean(), 6),
            "centroid_lon":     round(sub['GPS_LOG_DD'].mean(), 6),
            "lat_std":          round(sub['GPS_LAT_DD'].std(), 6),
            "lon_std":          round(sub['GPS_LOG_DD'].std(), 6),
            "max_internal_ft":  max_internal_ft,
            "avg_internal_ft":  avg_internal_ft,
            "flag": (
                "LARGE_CLUSTER"  if len(sub) > 10 else
                "SINGLETON"      if len(sub) == 1 else
                "OK"
            ),
        })

    summary_df = pd.DataFrame(summary_rows)
    out_path = clustered_filepath.replace('.csv', '_summary.csv')
    summary_df.to_csv(out_path, index=False)

    return json.dumps({
        "total_clusters":          len(summary_rows),
        "total_clustered_outlets": int(len(df_clusters)),
        "total_noise_outlets":     int((df['cluster_id'] == -1).sum()),
        "cluster_size_stats": {
            "mean":   round(summary_df['outlet_count'].mean(), 1),
            "median": round(summary_df['outlet_count'].median(), 1),
            "min":    int(summary_df['outlet_count'].min()),
            "max":    int(summary_df['outlet_count'].max()),
        },
        "flagged_clusters": {
            "large":      summary_df[summary_df['flag']=='LARGE_CLUSTER']['cluster_id'].tolist(),
            "singletons": summary_df[summary_df['flag']=='SINGLETON']['cluster_id'].tolist(),
        },
        "summary_csv": out_path,
        "per_cluster":  summary_rows,
    }, indent=2)

print("Tool 4: summarise_clusters")
r = json.loads(summarise_clusters("sample_outlets_clustered.csv"))
print(f"  Total clusters: {r['total_clusters']} | Size stats: {r['cluster_size_stats']}")



---

## Tool 5: Plot Cluster Map (Inline Visualisation)

**Concept:** Visual QA of the clustering result before exporting to QGIS.
Each cluster gets a distinct colour; noise outlets are shown in grey.
Cluster centroids are marked with a cross.


In [ ]:

@tool
def plot_clusters_map(clustered_filepath: str, title: str = None) -> str:
    """
    Plots a colour-coded scatter map of clustered outlets inline.
    Each cluster gets a distinct colour; noise outlets (cluster_id = -1)
    are shown in grey. Cluster centroids are marked with an X.
    Use this for visual QA before exporting to QGIS.

    Args:
        clustered_filepath: Path to clustered outlet CSV (output of cluster_outlets_dbscan)
        title:              Optional chart title (defaults to filename)

    Returns:
        Text confirmation that the chart was rendered.
    """
    from IPython.display import display as ipy_display

    df = pd.read_csv(clustered_filepath)
    if 'cluster_id' not in df.columns:
        return "cluster_id column not found. Run cluster_outlets_dbscan first."

    fig, ax = plt.subplots(figsize=(11, 9))

    cluster_ids = sorted(df[df['cluster_id'] != -1]['cluster_id'].unique())
    cmap        = cm.get_cmap('tab20', max(len(cluster_ids), 1))

    # Plot noise outlets
    noise = df[df['cluster_id'] == -1]
    if len(noise):
        ax.scatter(
            noise['GPS_LOG_DD'], noise['GPS_LAT_DD'],
            c='#cccccc', s=20, alpha=0.5, label='Noise (unclustered)', zorder=2
        )

    # Plot each cluster with a distinct colour
    for idx, cid in enumerate(cluster_ids):
        sub    = df[df['cluster_id'] == cid]
        colour = cmap(idx)
        ax.scatter(
            sub['GPS_LOG_DD'], sub['GPS_LAT_DD'],
            c=[colour], s=45, alpha=0.85, zorder=3,
            label=f'Cluster {cid} (n={len(sub)})',
        )
        # Mark centroid
        ax.scatter(
            sub['GPS_LOG_DD'].mean(), sub['GPS_LAT_DD'].mean(),
            marker='X', c=[colour], s=120, edgecolors='black',
            linewidths=0.8, zorder=5,
        )
        # Label cluster ID
        ax.annotate(
            str(cid),
            (sub['GPS_LOG_DD'].mean(), sub['GPS_LAT_DD'].mean()),
            textcoords='offset points', xytext=(5, 5),
            fontsize=8, fontweight='bold', color='black',
        )

    ax.set_xlabel('Longitude', fontsize=11)
    ax.set_ylabel('Latitude', fontsize=11)
    chart_title = title or f"Outlet Clusters — {os.path.basename(clustered_filepath)}"
    ax.set_title(chart_title, fontsize=13, fontweight='bold', pad=12)

    # Legend — show max 12 entries to avoid overflow
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:12], labels[:12], loc='lower right', fontsize=8,
              framealpha=0.85, title='Clusters')

    ax.grid(True, alpha=0.25, linestyle='--')
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=130)
    buf.seek(0)
    ipy_display(Image.open(buf))
    plt.close()

    return (f"Chart rendered: {len(cluster_ids)} clusters + "
            f"{len(noise)} noise outlets plotted.")

print("Tool 5: plot_clusters_map")
print(plot_clusters_map("sample_outlets_clustered.csv", title="Sample Outlet Clusters — Lagos"))



---

## Tool 6: Export for QGIS

**Concept:** QGIS is the standard open-source GIS tool used in NielsenIQ and retail
measurement teams for spatial QA of outlet sample frames. This tool produces three files:

| File | Format | Purpose |
|------|--------|---------|
| `_qgis.gpkg` | GeoPackage | Drag directly into QGIS — native vector format |
| `_qgis.csv` | CSV with WKT | Alternative import for QGIS "Delimited Text Layer" |
| `_style.qml` | QGIS Layer Style | Pre-built colour-coded symbology by cluster_id |

**QGIS workflow:**
1. Open QGIS → Layer → Add Layer → Add Vector Layer → select `.gpkg`
2. Right-click layer → Properties → Symbology → Style → Load Style → select `.qml`
3. Clusters are colour-coded automatically

For the full Python–QGIS integration, the optional cell below uses
`qgis.core` via PyQGIS (requires QGIS installed) to automate the project creation.


In [ ]:

@tool
def export_for_qgis(
    clustered_filepath: str,
    output_dir: str = ".",
) -> str:
    """
    Exports clustered outlet data in QGIS-compatible formats:
    GeoPackage (.gpkg) for direct QGIS drag-and-drop,
    CSV with WKT geometry for Delimited Text Layer import,
    and a QML style file for automatic colour-coded cluster symbology.

    Args:
        clustered_filepath: Path to clustered CSV (output of cluster_outlets_dbscan)
        output_dir:         Directory to write QGIS output files (default current dir)

    Returns:
        JSON string with paths to all exported files and QGIS import instructions.
    """
    df = pd.read_csv(clustered_filepath)
    if 'cluster_id' not in df.columns:
        return json.dumps({"error": "cluster_id column not found. Run cluster_outlets_dbscan first."})

    os.makedirs(output_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(clustered_filepath))[0]

    # ── Build GeoDataFrame ────────────────────────────────────────
    geometry = [Point(row['GPS_LOG_DD'], row['GPS_LAT_DD']) for _, row in df.iterrows()]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

    # Add human-readable cluster label
    gdf['cluster_label'] = gdf['cluster_id'].apply(
        lambda x: f'Cluster_{x}' if x != -1 else 'Noise'
    )

    # ── Export GeoPackage ─────────────────────────────────────────
    gpkg_path = os.path.join(output_dir, f"{base}_qgis.gpkg")
    gdf.to_file(gpkg_path, driver='GPKG', layer='outlets')

    # ── Export CSV with WKT ───────────────────────────────────────
    gdf['WKT'] = gdf['geometry'].apply(lambda p: f'POINT ({p.x} {p.y})')
    csv_path = os.path.join(output_dir, f"{base}_qgis.csv")
    gdf.drop(columns='geometry').to_csv(csv_path, index=False)

    # ── Generate QML style file ───────────────────────────────────
    cluster_ids = sorted(gdf[gdf['cluster_id'] != -1]['cluster_id'].unique())
    colours = [
        '#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
        '#a65628','#f781bf','#999999','#66c2a5','#fc8d62',
        '#8da0cb','#e78ac3','#a6d854','#ffd92f','#e5c494',
        '#b3b3b3','#1b9e77','#d95f02','#7570b3','#e7298a',
    ]

    # Build QML rule entries for each cluster
    rules_xml = ""
    for idx, cid in enumerate(cluster_ids):
        col  = colours[idx % len(colours)]
        rules_xml += f"""
        <rule label="Cluster {cid}" key="{{rule_{cid}}}" filter="&quot;cluster_id&quot; = {cid}">
          <symbol type="marker" name="0" alpha="1">
            <layer class="SimpleMarker">
              <prop k="color" v="{col[1:]}ff"/>
              <prop k="size" v="3"/>
              <prop k="outline_color" v="35,35,35,255"/>
              <prop k="outline_width" v="0.2"/>
            </layer>
          </symbol>
        </rule>"""

    # Noise rule
    rules_xml += """
        <rule label="Noise" key="{rule_noise}" filter="&quot;cluster_id&quot; = -1">
          <symbol type="marker" name="0" alpha="0.4">
            <layer class="SimpleMarker">
              <prop k="color" v="ccccccff"/>
              <prop k="size" v="2"/>
              <prop k="outline_color" v="aaaaaa255"/>
              <prop k="outline_width" v="0.1"/>
            </layer>
          </symbol>
        </rule>"""

    qml_content = f"""<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.28" styleCategories="Symbology">
  <renderer-v2 type="rule-based">
    <rules key="{{root}}">{rules_xml}
    </rules>
  </renderer-v2>
</qgis>
"""
    qml_path = os.path.join(output_dir, f"{base}_style.qml")
    with open(qml_path, 'w') as f:
        f.write(qml_content)

    return json.dumps({
        "exported_files": {
            "geopackage":  gpkg_path,
            "csv_wkt":     csv_path,
            "qml_style":   qml_path,
        },
        "crs":             "EPSG:4326 (WGS84)",
        "n_clusters_styled": len(cluster_ids),
        "total_outlets":   len(gdf),
        "qgis_instructions": [
            "1. Open QGIS",
            f"2. Layer → Add Layer → Add Vector Layer → select: {gpkg_path}",
            f"3. Right-click layer → Properties → Symbology → Style → Load Style → select: {qml_path}",
            "4. Clusters will be colour-coded automatically by cluster_id",
            "5. Alternatively: Layer → Add Delimited Text Layer → select the _qgis.csv file, set geometry from WKT column",
        ],
    }, indent=2)

print("Tool 6: export_for_qgis")
r = json.loads(export_for_qgis("sample_outlets_clustered.csv", output_dir="qgis_output"))
print(f"  Exported: {list(r['exported_files'].keys())}")
for step in r['qgis_instructions']:
    print(f"    {step}")



---

## Tool 7: Export Distance Pairs with Unique Cluster IDs

**What this fixes:** This directly replaces the output of the original `clustered_outlier.csv`
but does it correctly — unique cluster IDs assigned by DBSCAN, not by the broken
`defaultdict` loop. Every pair within the threshold gets the cluster ID of both outlets.


In [ ]:

@tool
def export_distance_pairs(
    clustered_filepath: str,
    threshold_miles: float = 0.284,
) -> str:
    """
    Exports all outlet pairs within the distance threshold with their cluster IDs.
    This is the corrected replacement for clustered_outlier.csv from the original script.
    Every row is one pair of outlets within threshold_miles of each other.
    Includes RESPONSEID, RESPONSEID_2, distance_miles, distance_feet, and cluster_id.

    Args:
        clustered_filepath: Path to clustered outlet CSV (output of cluster_outlets_dbscan)
        threshold_miles:    Maximum pair distance in miles (default 0.284 = 1500 feet)

    Returns:
        JSON string with pair count, output file path, and sample pairs.
    """
    df = pd.read_csv(clustered_filepath)
    if 'cluster_id' not in df.columns:
        return json.dumps({"error": "cluster_id column not found."})

    coords_rad = np.radians(df[['GPS_LAT_DD','GPS_LOG_DD']].values)
    dm         = DistanceMetric.get_metric('haversine')
    matrix     = dm.pairwise(coords_rad) * 3959.0   # miles

    ids = df['RESPONSEID'].values

    pair_rows = []
    # Loop over upper triangle only — avoids duplicate pairs
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            dist_miles = matrix[i, j]
            if dist_miles <= threshold_miles:      # if-then: apply threshold
                pair_rows.append({
                    'RESPONSEID':    ids[i],
                    'RESPONSEID_2':  ids[j],
                    'distance_miles': round(dist_miles, 6),
                    'distance_feet':  round(dist_miles * 5280, 2),
                    'cluster_id_1':  int(df['cluster_id'].iloc[i]),
                    'cluster_id_2':  int(df['cluster_id'].iloc[j]),
                    # Both outlets in same cluster if they match and aren't noise
                    'same_cluster':  (
                        df['cluster_id'].iloc[i] == df['cluster_id'].iloc[j]
                        and df['cluster_id'].iloc[i] != -1
                    ),
                })

    pairs_df = pd.DataFrame(pair_rows)
    out_path  = clustered_filepath.replace('.csv', '_distance_pairs.csv')
    pairs_df.to_csv(out_path, index=False)

    return json.dumps({
        "threshold_miles":     threshold_miles,
        "threshold_feet":      round(threshold_miles * 5280, 1),
        "total_pairs_found":   len(pairs_df),
        "same_cluster_pairs":  int(pairs_df['same_cluster'].sum()) if len(pairs_df) else 0,
        "cross_cluster_pairs": int((~pairs_df['same_cluster']).sum()) if len(pairs_df) else 0,
        "output_csv":          out_path,
        "sample_pairs":        pairs_df.head(5).to_dict(orient='records') if len(pairs_df) else [],
    }, indent=2)

print("Tool 7: export_distance_pairs")
r = json.loads(export_distance_pairs("sample_outlets_clustered.csv"))
print(f"  Pairs within 1500ft: {r['total_pairs_found']}")
print(f"  Same cluster: {r['same_cluster_pairs']} | Cross-cluster: {r['cross_cluster_pairs']}")



---

## Optional: Automate QGIS Project Creation via PyQGIS

If you have **QGIS installed on your machine**, the cell below uses the
`qgis.core` Python API (PyQGIS) to programmatically:
- Create a new QGIS project
- Load the GeoPackage layer
- Apply the QML style
- Save a `.qgs` project file you can open directly

> **Run this locally** (not in Colab) with QGIS Python environment active.
> In Colab, the cell is safe to skip — use the QGIS GUI instructions from Tool 6 instead.


In [ ]:

# ── PyQGIS automation (run locally with QGIS Python environment) ──────────
# Skip this cell in Colab — it requires a local QGIS installation.

try:
    from qgis.core import (
        QgsApplication, QgsProject, QgsVectorLayer,
        QgsLayerDefinition, QgsCoordinateReferenceSystem,
    )
    import sys

    # Initialise QGIS application (headless)
    qgs = QgsApplication([], False)
    qgs.initQgis()

    project   = QgsProject.instance()
    gpkg_path = "qgis_output/sample_outlets_clustered_qgis.gpkg"
    qml_path  = "qgis_output/sample_outlets_clustered_style.qml"

    # Load GeoPackage layer
    layer = QgsVectorLayer(f"{gpkg_path}|layername=outlets", "Outlet Clusters", "ogr")
    if not layer.isValid():
        print("Layer failed to load. Check the .gpkg path.")
    else:
        # Apply QML style
        layer.loadNamedStyle(qml_path)
        layer.triggerRepaint()

        # Add to project and set CRS
        project.addMapLayer(layer)
        project.setCrs(QgsCoordinateReferenceSystem("EPSG:4326"))

        # Save project
        project_path = "qgis_output/outlet_clusters.qgs"
        project.write(project_path)
        print(f"✅ QGIS project saved to: {project_path}")
        print("Open this file in QGIS to view the colour-coded cluster map.")

    qgs.exitQgis()

except ImportError:
    print("PyQGIS not available in this environment.")
    print("To use PyQGIS automation, run this cell in the QGIS Python Console or")
    print("a Python environment where QGIS is installed (qgis.core importable).")
    print("For Colab/Jupyter: use the QGIS GUI with the files exported by Tool 6.")



---

# Part 2: Single-Agent ReAct Demo

Watch the agent reason through the full clustering pipeline autonomously.


In [ ]:

clustering_agent = CodeAgent(
    tools=[
        load_and_validate_outlets,
        compute_haversine_distance_matrix,
        cluster_outlets_dbscan,
        summarise_clusters,
        plot_clusters_map,
        export_for_qgis,
        export_distance_pairs,
    ],
    model=model,
    verbosity_level=LogLevel.INFO,
    max_steps=20,
    additional_authorized_imports=[
        "pandas", "numpy", "sklearn", "sklearn.neighbors", "sklearn.cluster",
        "geopandas", "shapely", "shapely.geometry", "matplotlib",
        "matplotlib.pyplot", "matplotlib.cm", "scipy",
        "json", "os", "io", "PIL", "PIL.Image",
    ],
)
print("✅ Clustering agent ready — 7 tools loaded.")


In [ ]:

result = clustering_agent.run("""
Perform a complete outlet clustering analysis on 'sample_outlets.csv':
1. Load and validate the outlet data
2. Run DBSCAN clustering with 1500ft (0.284 miles) threshold, minimum 2 outlets per cluster
3. Summarise the clusters — how many were found, what are the sizes?
4. Plot the cluster map
5. Export all files for QGIS
6. Export the distance pairs CSV

Report: how many clusters were found, how many outlets are clustered vs noise,
and what files are ready for QGIS import.
""")
print(result)



---

# Part 3: Gradio Interface

A user-friendly interface for field operations teams who need to cluster outlets
without writing code. Upload a CSV, set the distance threshold, and click Run.


In [ ]:

import gradio as gr

def run_clustering_pipeline(
    uploaded_file,
    eps_miles: float,
    min_samples: int,
):
    """Gradio wrapper around the full clustering pipeline."""
    if uploaded_file is None:
        return "Please upload a CSV file.", "No file provided.", None

    # Gradio passes a NamedTemporaryFile
    filepath = uploaded_file.name

    log = []

    # Step 1: Validate
    log.append("🔍 Validating outlet data...")
    val  = json.loads(load_and_validate_outlets(filepath))
    if "error" in val:
        return f"❌ {val['error']}", "Validation failed.", None
    clean_fp = val['clean_filepath']
    log.append(f"  ✅ {val['valid_outlets']} valid outlets loaded.")

    # Step 2: Cluster
    log.append(f"📍 Clustering at {eps_miles:.3f} miles ({round(eps_miles*5280)} ft) threshold...")
    cl   = json.loads(cluster_outlets_dbscan(clean_fp, eps_miles=eps_miles, min_samples=min_samples))
    if "error" in cl:
        return f"❌ {cl['error']}", "Clustering failed.", None
    clustered_fp = cl['output_clustered']
    log.append(f"  ✅ {cl['n_clusters']} clusters | {cl['n_clustered_outlets']} clustered | {cl['n_noise_outlets']} noise")

    # Step 3: Summarise
    log.append("📊 Summarising clusters...")
    sm   = json.loads(summarise_clusters(clustered_fp))
    log.append(f"  ✅ Cluster sizes: {sm['cluster_size_stats']}")

    # Step 4: Plot (renders inline from tool)
    log.append("🗺️  Rendering cluster map...")
    plot_clusters_map(clustered_fp)
    log.append("  ✅ Map rendered above.")

    # Step 5: QGIS export
    log.append("📦 Exporting QGIS files...")
    qgis = json.loads(export_for_qgis(clustered_fp, output_dir="qgis_output"))
    log.append(f"  ✅ Exported: {', '.join(qgis['exported_files'].keys())}")

    # Step 6: Distance pairs
    log.append("📋 Exporting distance pairs...")
    dp   = json.loads(export_distance_pairs(clustered_fp, threshold_miles=eps_miles))
    log.append(f"  ✅ {dp['total_pairs_found']} pairs within threshold saved.")

    memo = f"""
## Outlet Clustering Results

---

### Data Summary
- **Valid outlets loaded:** {val['valid_outlets']}
- **Dropped (missing coords):** {val['dropped_missing']}
- **Coordinate bounds:** Lat [{val['coordinate_bounds']['lat_min']}, {val['coordinate_bounds']['lat_max']}]

### Clustering Parameters
- **Threshold:** {eps_miles} miles ({round(eps_miles*5280)} feet)
- **Min outlets per cluster:** {min_samples}
- **Algorithm:** DBSCAN with Haversine metric

### Results
- **Clusters found:** {cl['n_clusters']}
- **Clustered outlets:** {cl['n_clustered_outlets']} of {val['valid_outlets']}
- **Noise outlets:** {cl['n_noise_outlets']}
- **Cluster size range:** {sm['cluster_size_stats']['min']}–{sm['cluster_size_stats']['max']} outlets

### QGIS Export
- GeoPackage: `{qgis['exported_files']['geopackage']}`
- CSV (WKT): `{qgis['exported_files']['csv_wkt']}`
- Style file: `{qgis['exported_files']['qml_style']}`

### Distance Pairs
- **Pairs within threshold:** {dp['total_pairs_found']}
- **Same-cluster pairs:** {dp['same_cluster_pairs']}
- **Cross-cluster pairs:** {dp['cross_cluster_pairs']}

---
*Drag the .gpkg file into QGIS and load the .qml style for colour-coded cluster visualisation.*
"""
    return memo, "\n".join(log)


with gr.Blocks(
    title="Outlet Clustering Agent",
    theme=gr.themes.Soft(),
) as demo:

    gr.HTML("""
        <div style='text-align:center;padding:16px;background:#1C7293;
                    color:white;border-radius:8px;margin-bottom:16px'>
            <h2>📍 Outlet Clustering Agent</h2>
        </div>
        <p style='text-align:center;color:#666;font-size:13px;margin-bottom:16px'>
            Haversine · DBSCAN · QGIS Export · Gemini 2.5 Flash<br>
            Upload an outlet CSV with RESPONSEID, GPS_LAT_DD, GPS_LOG_DD columns
        </p>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            file_upload = gr.File(
                label="📂 Upload Outlet CSV",
                file_types=[".csv"],
            )
            eps_slider = gr.Slider(
                minimum=0.05, maximum=1.0, step=0.01, value=0.284,
                label="Cluster Threshold (miles) — 0.284 = 1500 ft",
            )
            min_samp = gr.Slider(
                minimum=2, maximum=10, step=1, value=2,
                label="Minimum Outlets per Cluster",
            )
            run_btn = gr.Button("▶ Run Clustering Pipeline", variant="primary", size="lg")
            status_box = gr.Textbox(label="⚙️ Status Log", lines=10, interactive=False)
            gr.HTML("""
                <p style='font-size:11px;color:#999;margin-top:8px'>
                Required columns: RESPONSEID, GPS_LAT_DD, GPS_LOG_DD
                </p>
            """)

        with gr.Column(scale=2):
            output_box = gr.Markdown(
                value="*Upload a CSV and click Run Clustering Pipeline.*"
            )

    run_btn.click(
        fn=run_clustering_pipeline,
        inputs=[file_upload, eps_slider, min_samp],
        outputs=[output_box, status_box],
    )

print("✅ Gradio interface defined. Launching...")
demo.launch(share=True, debug=False)



---

# Discussion: Original Script vs. Agentic Approach

## Bugs Fixed from the 2020 Script

```python
# ORIGINAL — 5 bugs in this block alone:
for i in range(len(colors, 0)):         # Bug 1: len() takes 1 arg, not 2
    unique=[]                           # Bug 2: resets unique list every iteration
    if colours[i] not in unique:        # Bug 3: typo — 'colours' not defined
        unique=np.append(unique,ratios[(i)])
temp = defaultdict(lambda: len(temp))
ru_dict= dict(zip(ru,res))
        nc['unique_id'] = nc.id2.map(unique_dict)  # Bug 4: wrong indentation
        i = i+1                                     # Bug 5: manual increment in for-loop
    else:
        continue

# FIXED — replaced entirely by DBSCAN cluster labels:
df['cluster_id'] = DBSCAN(eps=eps_rad, metric='haversine').fit(coords_rad).labels_
```

## Traditional GIS Workflow vs. Agentic

| Step | Traditional (ArcGIS/QGIS + Excel) | This Agent |
|------|-----------------------------------|-----------------------|
| Load & validate data | Manual CSV review, Excel filter | `load_and_validate_outlets()` |
| Compute distances | Python script or QGIS Distance Matrix tool | `compute_haversine_distance_matrix()` |
| Assign clusters | Manual classification or QGIS DBSCAN plugin | `cluster_outlets_dbscan()` |
| Summary statistics | Excel pivot table | `summarise_clusters()` |
| Visualise | QGIS → style → export | `plot_clusters_map()` inline |
| Export to GIS | Save as Shapefile manually | `export_for_qgis()` → .gpkg + .qml |
| Distance pair report | Manual filtering in Excel | `export_distance_pairs()` |
| **Total time** | **2–4 hours per dataset** | **< 2 minutes** |

## Why QGIS + Python Agent is the Right Combination

**Python handles the computation.** Haversine distance matrices, DBSCAN clustering,
and statistical summaries are far more efficient in `scikit-learn` than in QGIS plugins
— especially at 10,000+ outlet scale.

**QGIS handles the spatial QA.** The `.gpkg` + `.qml` outputs give GIS analysts a
fully styled, interactive map they can interrogate — zoom, query, filter — without
writing any code. The QML style file means they don't have to re-classify clusters
manually every time.

**The agent handles the orchestration.** A field operations manager can describe the
problem in plain English through the Gradio interface and receive QGIS-ready files
without knowing Python, QGIS, or DBSCAN.

## Responsible Use in Retail Measurement

- **Coordinate privacy:** Outlet GPS coordinates may constitute sensitive business data.
  Ensure files are stored and transmitted in compliance with data sharing agreements.
- **Threshold sensitivity:** The 1500ft / 0.284 mile threshold is a business rule, not
  a statistical law. Always validate cluster assignments against ground truth before
  using them to modify a retail sample frame.
- **Noise outlets:** DBSCAN labels some outlets as noise (cluster_id = -1).
  These are not errors — they are genuinely isolated outlets that should be reviewed
  individually rather than merged into a nearby cluster.
